## Actividad 3_11

<div style="border-style:groove;border-width:thin;padding:10px">

Trabajamos en una empresa de alquiler de bicicletas en Seul. El departamento de mantenimiento de bicicletas ha pedido a nuestro departamento de DataScience si es posible poder saber, a priori, cuantas bicicletas tiene que haber disponibles porque ellos tienen que ir retirando bicicletas para repararlas. Tenemos los datos que se muestran a continuación. ¿Será posible saber cuantas tienen que tener en circulación?

</div>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_moons, make_circles
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_graphviz, DecisionTreeRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, r2_score, mean_squared_error

In [2]:
import pandas as pd
bicicletas = pd.read_csv("SeoulBikeData.csv", sep=',',encoding='latin-1')
bicicletas.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [3]:
bicicletas.drop(columns=["Date"], inplace=True)

In [4]:
bicicletas.info()
bicicletas.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Rented Bike Count          8760 non-null   int64  
 1   Hour                       8760 non-null   int64  
 2   Temperature(°C)            8760 non-null   float64
 3   Humidity(%)                8760 non-null   int64  
 4   Wind speed (m/s)           8760 non-null   float64
 5   Visibility (10m)           8760 non-null   int64  
 6   Dew point temperature(°C)  8760 non-null   float64
 7   Solar Radiation (MJ/m2)    8760 non-null   float64
 8   Rainfall(mm)               8760 non-null   float64
 9   Snowfall (cm)              8760 non-null   float64
 10  Seasons                    8760 non-null   object 
 11  Holiday                    8760 non-null   object 
 12  Functioning Day            8760 non-null   object 
dtypes: float64(6), int64(4), object(3)
memory usage:

(8760, 13)

In [5]:
bicicletas.head()

,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [6]:
bicicletas = pd.get_dummies(bicicletas, columns=["Seasons", "Holiday"], dtype=int)

In [7]:
bicicletas['Functioning Day'] = bicicletas['Functioning Day'].replace({'Yes': 1, 'No': 0})

C:\Users\Usuario\AppData\Local\Temp\ipykernel_1780\1704255731.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bicicletas['Functioning Day'] = bicicletas['Functioning Day'].replace({'Yes': 1, 'No': 0})


In [8]:
bicicletas.corr(numeric_only=True)['Rented Bike Count'].abs().sort_values(ascending=False)[1:]

Temperature(°C)              0.538558
Seasons_Winter               0.424925
Hour                         0.410257
Dew point temperature(°C)    0.379788
Seasons_Summer               0.296549
Solar Radiation (MJ/m2)      0.261837
Functioning Day              0.203943
Humidity(%)                  0.199780
Visibility (10m)             0.199280
Snowfall (cm)                0.141804
Rainfall(mm)                 0.123074
Wind speed (m/s)             0.121108
Seasons_Autumn               0.102753
Holiday_Holiday              0.072338
Holiday_No Holiday           0.072338
Seasons_Spring               0.022888
Name: Rented Bike Count, dtype: float64

In [9]:
X = bicicletas.drop(['Rented Bike Count'],axis=1)
y = bicicletas['Rented Bike Count'].to_frame()

In [10]:
escalador = StandardScaler()
X = escalador.fit_transform(X)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)

In [12]:
tree_clf = DecisionTreeRegressor(min_samples_split=50)
tree_clf.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",50
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"ma

In [13]:
y_pred = tree_clf.predict(X_test)

In [14]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

MSE: 82101.6403
RMSE: 286.5338
MAE: 172.5247
R2: 0.8029


In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

def mejor_combinacion_grid_search(tipo='regressor'):
    """
    Encuentra la mejor combinación usando GridSearchCV (prueba TODAS las combinaciones).
    """
    # Definir el espacio de búsqueda
    param_grid = {
        'max_depth': [3, 5, 10, 15, 20, None],
        'min_samples_split': [2, 10, 50, 100],
        'min_samples_leaf': [1, 5, 20, 50],
        'max_leaf_nodes': [None, 8, 16, 32, 64]
    }
    
    if tipo == 'regressor':
        param_grid['criterion'] = ['squared_error', 'friedman_mse', 'absolute_error']
        model = DecisionTreeRegressor(random_state=42)
        scoring = 'r2'
    else:
        param_grid['criterion'] = ['gini', 'entropy']
        model = DecisionTreeClassifier(random_state=42)
        scoring = 'accuracy'
    
    print(f"\n{'='*80}")
    print(f"GRID SEARCH - {tipo.upper()}")
    print(f"Total de combinaciones a probar: {6 * 3 * 4 * 4 * 5}")
    print(f"{'='*80}\n")
    
    # Realizar Grid Search
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        cv=5,  # Validación cruzada con 5 folds
        n_jobs=-1,  # Usar todos los procesadores
        verbose=2
    )
    
    grid_search.fit(X_train, y_train)
    
    # Resultados
    print(f"\n{'='*80}")
    print(f"MEJOR COMBINACIÓN ENCONTRADA (Grid Search)")
    print(f"{'='*80}")
    print(f"Mejor score (CV): {grid_search.best_score_:.4f}")
    print(f"\nHiperparámetros:")
    for param, valor in grid_search.best_params_.items():
        print(f"  - {param:20}: {valor}")
    
    # Evaluar en test
    y_pred = grid_search.best_estimator_.predict(X_test)
    if tipo == 'regressor':
        test_score = r2_score(y_test, y_pred)
        print(f"\nR² en test: {test_score:.4f}")
    else:
        test_score = accuracy_score(y_test, y_pred)
        print(f"\nAccuracy en test: {test_score:.4f}")
    
    print(f"{'='*80}\n")
    
    return grid_search.best_params_, grid_search.best_estimator_


# ========== USO ==========

# Para REGRESIÓN:
mejores_params, mejor_modelo = mejor_combinacion_grid_search(tipo='regressor')

# Para CLASIFICACIÓN:
# mejores_params, mejor_modelo = mejor_combinacion_grid_search(tipo='classifier')


GRID SEARCH - REGRESSOR
Total de combinaciones a probar: 1440

Fitting 5 folds for each of 1440 candidates, totalling 7200 fits

MEJOR COMBINACIÓN ENCONTRADA (Grid Search)
Mejor score (CV): 0.8214

Hiperparámetros:
  - criterion           : squared_error
  - max_depth           : 15
  - max_leaf_nodes      : None
  - min_samples_leaf    : 5
  - min_samples_split   : 50

R² en test: 0.8013



In [18]:
# Definir valores a probar para cada hiperparámetro
valores_max_depth = [3, 5, 10, 15, 20, None]
valores_criterion_clf = ['gini', 'entropy']
valores_criterion_reg = ['squared_error', 'friedman_mse', 'absolute_error']
valores_min_samples_split = [2, 10, 50, 100]
valores_min_samples_leaf = [1, 5, 20, 50]
valores_max_leaf_nodes = [None, 8, 16, 32, 64]

def mejor_eleccion_tree(hiperparametro, tipo='regressor'):
    if tipo == 'regressor':
        mejor_metrica = -float('inf')  # R² (cuanto más alto mejor)
        metrica_nombre = "R²"
    else:  # classifier
        mejor_metrica = 0  # Accuracy (cuanto más alto mejor)
        metrica_nombre = "Accuracy"
    
    mejor_config = None
    
    # MAX_DEPTH
    if hiperparametro == 'max_depth':
        for max_depth in valores_max_depth:
            print(f"\nProbando max_depth={max_depth}")
            
            if tipo == 'regressor':
                model = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
            else:
                model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
            
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            
            if tipo == 'regressor':
                metrica = r2_score(y_test, pred)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                mae = mean_absolute_error(y_test, pred)
                print(f"R²: {metrica:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
            else:
                metrica = accuracy_score(y_test, pred)
                print(f"Accuracy: {metrica:.4f}")
            
            if metrica > mejor_metrica:
                mejor_metrica = metrica
                mejor_config = ('max_depth', max_depth)
    
    # CRITERION
    elif hiperparametro == 'criterion':
        criterios = valores_criterion_reg if tipo == 'regressor' else valores_criterion_clf
        
        for criterion in criterios:
            print(f"\nProbando criterion={criterion}")
            
            if tipo == 'regressor':
                model = DecisionTreeRegressor(criterion=criterion, random_state=42)
            else:
                model = DecisionTreeClassifier(criterion=criterion, random_state=42)
            
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            
            if tipo == 'regressor':
                metrica = r2_score(y_test, pred)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                mae = mean_absolute_error(y_test, pred)
                print(f"R²: {metrica:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
            else:
                metrica = accuracy_score(y_test, pred)
                print(f"Accuracy: {metrica:.4f}")
            
            if metrica > mejor_metrica:
                mejor_metrica = metrica
                mejor_config = ('criterion', criterion)
    
    # MIN_SAMPLES_SPLIT
    elif hiperparametro == 'min_samples_split':
        for min_samples_split in valores_min_samples_split:
            print(f"\nProbando min_samples_split={min_samples_split}")
            
            if tipo == 'regressor':
                model = DecisionTreeRegressor(min_samples_split=min_samples_split, random_state=42)
            else:
                model = DecisionTreeClassifier(min_samples_split=min_samples_split, random_state=42)
            
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            
            if tipo == 'regressor':
                metrica = r2_score(y_test, pred)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                mae = mean_absolute_error(y_test, pred)
                print(f"R²: {metrica:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
            else:
                metrica = accuracy_score(y_test, pred)
                print(f"Accuracy: {metrica:.4f}")
            
            if metrica > mejor_metrica:
                mejor_metrica = metrica
                mejor_config = ('min_samples_split', min_samples_split)
    
    # MIN_SAMPLES_LEAF
    elif hiperparametro == 'min_samples_leaf':
        for min_samples_leaf in valores_min_samples_leaf:
            print(f"\nProbando min_samples_leaf={min_samples_leaf}")
            
            if tipo == 'regressor':
                model = DecisionTreeRegressor(min_samples_leaf=min_samples_leaf, random_state=42)
            else:
                model = DecisionTreeClassifier(min_samples_leaf=min_samples_leaf, random_state=42)
            
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            
            if tipo == 'regressor':
                metrica = r2_score(y_test, pred)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                mae = mean_absolute_error(y_test, pred)
                print(f"R²: {metrica:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
            else:
                metrica = accuracy_score(y_test, pred)
                print(f"Accuracy: {metrica:.4f}")
            
            if metrica > mejor_metrica:
                mejor_metrica = metrica
                mejor_config = ('min_samples_leaf', min_samples_leaf)
    
    # MAX_LEAF_NODES
    elif hiperparametro == 'max_leaf_nodes':
        for max_leaf_nodes in valores_max_leaf_nodes:
            print(f"\nProbando max_leaf_nodes={max_leaf_nodes}")
            
            if tipo == 'regressor':
                model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=42)
            else:
                model = DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=42)
            
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            
            if tipo == 'regressor':
                metrica = r2_score(y_test, pred)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                mae = mean_absolute_error(y_test, pred)
                print(f"R²: {metrica:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")
            else:
                metrica = accuracy_score(y_test, pred)
                print(f"Accuracy: {metrica:.4f}")
            
            if metrica > mejor_metrica:
                mejor_metrica = metrica
                mejor_config = ('max_leaf_nodes', max_leaf_nodes)
    
    print(f"\n{'='*60}")
    print(f"Mejor {metrica_nombre}: {mejor_metrica:.4f}")
    print(f"Mejor configuración: {mejor_config}")
    print(f"{'='*60}")
    
    return mejor_config, mejor_metrica


# ========== USO ==========

# Para REGRESIÓN:
mejor_eleccion_tree('max_depth', tipo='regressor')
mejor_eleccion_tree('criterion', tipo='regressor')
mejor_eleccion_tree('min_samples_split', tipo='regressor')
mejor_eleccion_tree('min_samples_leaf', tipo='regressor')
mejor_eleccion_tree('max_leaf_nodes', tipo='regressor')

# Para CLASIFICACIÓN:
# mejor_eleccion_tree('max_depth', tipo='classifier')
# mejor_eleccion_tree('criterion', tipo='classifier')
# mejor_eleccion_tree('min_samples_split', tipo='classifier')
# mejor_eleccion_tree('min_samples_leaf', tipo='classifier')
# mejor_eleccion_tree('max_leaf_nodes', tipo='classifier')


Probando max_depth=3
R²: 0.5347 | RMSE: 440.2897 | MAE: 294.4921

Probando max_depth=5
R²: 0.7034 | RMSE: 351.5240 | MAE: 226.3131

Probando max_depth=10
R²: 0.7784 | RMSE: 303.8400 | MAE: 177.0762

Probando max_depth=15
R²: 0.7260 | RMSE: 337.8647 | MAE: 188.5835

Probando max_depth=20
R²: 0.7195 | RMSE: 341.8673 | MAE: 192.9363

Probando max_depth=None
R²: 0.7099 | RMSE: 347.6516 | MAE: 196.7443

Mejor R²: 0.7784
Mejor configuración: ('max_depth', 10)

Probando criterion=squared_error
R²: 0.7099 | RMSE: 347.6516 | MAE: 196.7443

Probando criterion=friedman_mse
R²: 0.7099 | RMSE: 347.6516 | MAE: 196.7443

Probando criterion=absolute_error
R²: 0.7551 | RMSE: 319.4558 | MAE: 181.4121

Mejor R²: 0.7551
Mejor configuración: ('criterion', 'absolute_error')

Probando min_samples_split=2
R²: 0.7099 | RMSE: 347.6516 | MAE: 196.7443

Probando min_samples_split=10
R²: 0.7514 | RMSE: 321.8388 | MAE: 184.5381

Probando min_samples_split=50
R²: 0.8029 | RMSE: 286.5338 | MAE: 172.5247

Probando mi

(('max_leaf_nodes', 64), 0.7872332753246727)